In [2]:
import os

# Trouve le dossier racine du projet en remontant depuis le dossier courant jusqu'à ce qu'on trouve 'data'
def find_project_root(current_path, target_folder="data"):
    path = current_path
    while True:
        if target_folder in os.listdir(path):
            return path
        new_path = os.path.dirname(path)
        if new_path == path:  # On est arrivé à la racine du disque sans trouver
            raise FileNotFoundError(f"Le dossier '{target_folder}' n'a pas été trouvé dans la hiérarchie des dossiers.")
        path = new_path

# Récupère le dossier courant
current_dir = os.getcwd()

# Trouve la racine du projet (dossier qui contient 'data')
project_root = find_project_root(current_dir, target_folder="data")

# Construit le chemin vers le dossier data/processed
data_processed_dir = os.path.join(project_root, "data", "processed")

# Chemin complet vers le fichier CSV
csv_path = os.path.join(data_processed_dir, "accidents_clean.csv")

print("Chemin absolu vers le fichier :", csv_path)

# Chargement du fichier
import pandas as pd
df = pd.read_csv(csv_path)



Chemin absolu vers le fichier : /Users/alizeeblanchon/Documents/Data_Scientist/data_project/mai25_bds_accidents/data/processed/accidents_clean.csv


In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTETomek
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import MultiLabelBinarizer
import seaborn as sns
import matplotlib.pyplot as plt

# === 1. Chargement des données


X = df.drop(columns=['grav'])
y = df['grav']

# === 2. Binarisation multilabel sur 'equipements'
mlb = MultiLabelBinarizer()
equip = pd.DataFrame(mlb.fit_transform(X['equipements']),
                     columns=[f'eq_{str(c)}' for c in mlb.classes_],
                     index=X.index)
X = pd.concat([X.drop(columns='equipements'), equip], axis=1)

# === 3. Préprocessing
equip_cols = list(equip.columns)
cols_to_exclude = ['dep'] + equip_cols
cols_to_ohe = [col for col in X.columns if col not in cols_to_exclude]

preprocessor = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cols_to_ohe),
    ('target_enc', TargetEncoder(), ['dep'])
], remainder='passthrough')

# === 4. Pipeline avec SMOTETomek + StandardScaler
pipeline = ImbPipeline(steps=[
    ('encoding', preprocessor),
    ('scaling', StandardScaler()),  # Important pour LogisticRegression
    ('sampling', SMOTETomek(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs'))
])

# === 5. Validation croisée
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred = cross_val_predict(pipeline, X, y, cv=cv)

# === 6. Évaluation
print("F1-score pondéré :", f1_score(y, y_pred, average='weighted'))
print("\nClassification Report :\n", classification_report(y, y_pred))

# === 7. Matrice de confusion
cm = confusion_matrix(y, y_pred)
classes = sorted(y.unique())
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion - Validation croisée")
plt.show()


KeyError: 'equipements'

In [ ]:
#Remplacement de SmoteTomek par RandomUnderSampler
# === 1. Chargement des données

X = df.drop(columns=['grav'])
y = df['grav']

# === 2. Binarisation multilabel sur 'equipements'
mlb = MultiLabelBinarizer()
equip = pd.DataFrame(mlb.fit_transform(X['equipements']),
                     columns=[f'eq_{str(c)}' for c in mlb.classes_],
                     index=X.index)
X = pd.concat([X.drop(columns='equipements'), equip], axis=1)

# === 3. Préprocessing
equip_cols = list(equip.columns)
cols_to_exclude = ['dep'] + equip_cols
cols_to_ohe = [col for col in X.columns if col not in cols_to_exclude]

preprocessor = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cols_to_ohe),
    ('target_enc', TargetEncoder(), ['dep'])
], remainder='passthrough')

# === 4. Pipeline avec RandomUnderSampler
pipeline = ImbPipeline(steps=[
    ('encoding', preprocessor),
    ('scaling', StandardScaler()),
    ('sampling', RandomUnderSampler(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs'))
])

# === 5. Validation croisée
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred = cross_val_predict(pipeline, X, y, cv=cv)

# === 6. Évaluation
print("F1-score pondéré :", f1_score(y, y_pred, average='weighted'))
print("\nClassification Report :\n", classification_report(y, y_pred))

# === 7. Matrice de confusion
cm = confusion_matrix(y, y_pred)
classes = sorted(y.unique())
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion - Validation croisée avec sous-échantillonnage")
plt.show()
